# IPO 관련 테스트 코딩
IPO와 관련하여 엑셀파일 작성이나 수요예측 참여의 자동화와 관련된 작업을 하게 될 테스트 파일

## 1. 수요예측 엑셀파일의 시트 작성
- 엑셀파일에 이미 작성되어 있는 각 수요예측 종목별 시트를 참고하여 이미 작성되어 있는 종목은 제외하고 38커뮤니케이션의 수요예측 일정 페이지의 각 종목별 페이지에서 [종목명, 종목코드, 시장구분, 희망공모가액, 수요예측일, 공모청약일, 기관투자자 최대배정수량(공모사항-그룹별배정-기관투자자등)]을 크롤링.
    크롤링을 시작할 38커뮤니케이션의 페이지는 'https://www.38.co.kr/html/fund/index.htm?o=r'
- 참고할 엑셀파일은 [Z:\02.펀드\001.수요예측\기타\수요예측일정.xlsx]경로에 있음. 코드는 엑셀파일을 참고하여 새로운 시트를 복사하여 만들고 크롤링한 데이터를 키워드에 맞게 입력.
- 수요예측 파일은 매일매일 크롤링해서 업데이트 하되 엑셀파일의 용량이 비대해지면 로딩이 오래걸릴 것을 염려하여 분기마다 다른 엑셀파일로 자동생성할 것.
    수요예측 파일 이름은 YYYY"Q"N으로 생성할 것. 예시) 2026Q1, 2026Q2


### 1) 38커뮤니케이션 수요예측 일정 페이지 크롤링
1. 'https://www.38.co.kr/html/fund/index.htm?o=r'로 이동
2. 종목명 컬럼을 찾아서 엑셀시트에 없는 종목만 찾아서 종목명을 클릭하여 페이지 이동
3. 이동한 페이지에서 [종목명, 종목코드, 시장구분, 희망공모가액, 수요예측일, 공모청약일, 기관투자자 최대배정수량(공모사항-그룹별배정-기관투자자등), 주간사] 정보 크롤링하여 [Z:\02.펀드\001.수요예측\기타\수요예측일정.xlsx]의 [IPO정보]시트에 각 칼럼에 맞는 정보를 기록.
4. [수요예측일]이 가장 빠른 종목이 가장 위로 오도록 정렬.

In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

def get_ipo_schedule():
    url = 'https://www.38.co.kr/html/fund/index.htm?o=r'
    
    # 1. 브라우저처럼 보이게 하기 위한 헤더 설정 (차단 방지)
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    # 2. 페이지 요청
    response = requests.get(url, headers=headers)
    response.encoding = 'euc-kr'  # 38커뮤니케이션은 한글 인코딩이 euc-kr인 경우가 많습니다.

    # 3. Pandas로 표 데이터 읽기
    # [0]번 혹은 [1]번 등 인덱스를 변경하며 원하는 표를 찾습니다.
    dfs = pd.read_html(response.text)
    
    # 보통 본문 표는 데이터가 가장 많은 프레임에 위치합니다.
    # 실무적으로는 BeautifulSoup으로 특정 클래스나 id를 집어내는 것이 더 정확합니다.
    return dfs

# 실행 예시
all_tables = get_ipo_schedule()
# 데이터가 있는 표 확인 (보통 인덱스 조절이 필요합니다)
for i, df in enumerate(all_tables):
    print(f"--- Table {i} ---")
    print(df.head())

SSLError: HTTPSConnectionPool(host='www.38.co.kr', port=443): Max retries exceeded with url: /html/fund/index.htm?o=r (Caused by SSLError(SSLError(1, '[SSL: SSLV3_ALERT_HANDSHAKE_FAILURE] sslv3 alert handshake failure (_ssl.c:1032)')))